In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import glob
import os


In [2]:
class GeneradorMapasInteligentes:
    """Generador de mapas de calor optimizados para visualización"""

    def __init__(self, ruta_resultados='../../Limpieza/data/resultados', debug=True):
        self.ruta = ruta_resultados
        self.debug = debug
        self.datos_cargados = None
        self.mapas_generados = []

    def log(self, mensaje):
        if self.debug:
            print(f"[INFO] {mensaje}")

    def cargar_datos(self):
        """Carga y analiza los datos disponibles"""
        json_files = glob.glob(os.path.join(self.ruta, '*_results.json'))

        if len(json_files) == 0:
            print("❌ No se encontraron archivos JSON")
            return False

        datos = []
        for json_file in json_files:
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    datos.append(data)
            except Exception as e:
                if self.debug:
                    print(f"Error con {os.path.basename(json_file)}: {str(e)}")

        self.datos_cargados = datos
        self.log(f"✅ Cargados {len(datos)} archivos")

        # Analizar qué datos tenemos
        self.analizar_datos_disponibles()

        return len(datos) > 0

    def analizar_datos_disponibles(self):
        """Analiza qué tipos de datos tenemos disponibles"""
        municipios_con_velocidades = []
        municipios_con_tecnologias = []
        tecnologias_encontradas = set()
        velocidades_bajada = []

        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')

            # Análisis de velocidades
            if ('velocidades' in data and
                data['velocidades'].get('trends') and
                'bajada' in data['velocidades']['trends']):

                vel_bajada = data['velocidades']['trends']['bajada'].get('media', 0)
                if vel_bajada > 0:
                    municipios_con_velocidades.append(municipio)
                    velocidades_bajada.append(vel_bajada)

            # Análisis de tecnologías
            if 'tecnologias' in data and data['tecnologias'].get('trends'):
                municipios_con_tecnologias.append(municipio)
                for tech in data['tecnologias']['trends'].keys():
                    tecnologias_encontradas.add(tech)

        print(f"\n📊 ANÁLISIS DE DATOS DISPONIBLES:")
        print(f"   • Total municipios: {len(self.datos_cargados)}")
        print(f"   • Con datos de velocidades: {len(municipios_con_velocidades)}")
        print(f"   • Con datos de tecnologías: {len(municipios_con_tecnologias)}")
        print(f"   • Tecnologías encontradas: {len(tecnologias_encontradas)}")
        print(f"   • Tecnologías: {sorted(list(tecnologias_encontradas))}")

        if velocidades_bajada:
            print(f"   • Velocidad promedio: {np.mean(velocidades_bajada):.1f} Mbps")
            print(f"   • Rango velocidades: {np.min(velocidades_bajada):.1f} - {np.max(velocidades_bajada):.1f} Mbps")

        # Sugerencias de visualización
        print(f"\n💡 SUGERENCIAS DE VISUALIZACIÓN:")
        if len(municipios_con_velocidades) > 50:
            print(f"   • Demasiados municipios ({len(municipios_con_velocidades)}) - usaremos TOP 20")
        if len(tecnologias_encontradas) > 8:
            print(f"   • Muchas tecnologías ({len(tecnologias_encontradas)}) - agruparemos similares")

        return {
            'municipios_velocidades': len(municipios_con_velocidades),
            'municipios_tecnologias': len(municipios_con_tecnologias),
            'tecnologias': list(tecnologias_encontradas),
            'velocidades_promedio': np.mean(velocidades_bajada) if velocidades_bajada else 0
        }

    def generar_top_municipios_velocidad(self, top_n=20, incluir_peores=5):
        """Genera heatmap con los mejores Y peores municipios"""
        self.log(f"📊 Generando TOP {top_n} municipios por velocidad...")

        # Recopilar datos de velocidades
        municipios_data = []
        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')

            if ('velocidades' in data and
                data['velocidades'].get('trends')):

                vel_trends = data['velocidades']['trends']

                velocidad_bajada = vel_trends.get('bajada', {}).get('media', 0)
                velocidad_subida = vel_trends.get('subida', {}).get('media', 0)
                r2_bajada = vel_trends.get('bajada', {}).get('r2', 0)

                if velocidad_bajada > 0:  # Solo municipios con datos válidos
                    municipios_data.append({
                        'municipio': municipio,
                        'vel_bajada': velocidad_bajada,
                        'vel_subida': velocidad_subida,
                        'r2_bajada': r2_bajada,
                        'ratio_subida_bajada': velocidad_subida / velocidad_bajada if velocidad_bajada > 0 else 0
                    })

        if len(municipios_data) == 0:
            self.log("⚠️ No hay datos de velocidades válidos")
            return None

        # Crear DataFrame y ordenar
        df = pd.DataFrame(municipios_data)
        df_sorted = df.sort_values('vel_bajada', ascending=False)

        # Tomar top N y peores incluir_peores
        df_top = df_sorted.head(top_n)
        if incluir_peores > 0 and len(df_sorted) > top_n:
            df_bottom = df_sorted.tail(incluir_peores)
            df_final = pd.concat([df_top, df_bottom])
        else:
            df_final = df_top

        # Preparar datos para heatmap
        heatmap_data = df_final[['vel_bajada', 'vel_subida', 'r2_bajada', 'ratio_subida_bajada']].set_index(df_final['municipio'])
        heatmap_data.columns = ['Velocidad Bajada\n(Mbps)', 'Velocidad Subida\n(Mbps)',
                               'Calidad Predicción\n(R²)', 'Ratio Subida/Bajada']

        # Normalizar para mejor visualización (excepto R² que ya está 0-1)
        heatmap_normalized = heatmap_data.copy()
        for col in ['Velocidad Bajada\n(Mbps)', 'Velocidad Subida\n(Mbps)', 'Ratio Subida/Bajada']:
            if col in heatmap_normalized.columns:
                max_val = heatmap_normalized[col].max()
                if max_val > 0:
                    heatmap_normalized[col] = heatmap_normalized[col] / max_val

        # Crear el heatmap
        plt.figure(figsize=(10, max(8, len(heatmap_data.index) * 0.5)))

        sns.heatmap(heatmap_normalized,
                    annot=heatmap_data,  # Mostrar valores reales
                    cmap='RdYlGn',
                    fmt='.1f',
                    cbar_kws={'shrink': .8, 'label': 'Valor Normalizado'},
                    linewidths=0.5)

        plt.title(f'🏆 TOP {top_n} Municipios por Velocidad de Internet\n(Incluye {incluir_peores} peores para comparación)',
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel('Métricas', fontsize=12)
        plt.ylabel('Municipio', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0, fontsize=10)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, f'top_{top_n}_municipios_velocidad.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_tecnologias_principales(self, top_tecnologias=6, top_municipios=15):
        """Genera heatmap con las principales tecnologías y municipios"""
        self.log(f"📊 Generando mapa: TOP {top_tecnologias} tecnologías x TOP {top_municipios} municipios...")

        # Recopilar datos de tecnologías
        tech_municipio_data = []
        tech_stats = {}  # Para calcular promedios por tecnología

        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')

            if 'tecnologias' in data and data['tecnologias'].get('trends'):
                for tech, trends in data['tecnologias']['trends'].items():
                    velocidad = trends.get('velocidad_media', 0)
                    accesos = trends.get('accesos_media', 0)

                    if velocidad > 0:  # Solo datos válidos
                        tech_municipio_data.append({
                            'municipio': municipio,
                            'tecnologia': tech,
                            'velocidad': velocidad,
                            'accesos': accesos
                        })

                        # Acumular stats por tecnología
                        if tech not in tech_stats:
                            tech_stats[tech] = {'velocidades': [], 'accesos': []}
                        tech_stats[tech]['velocidades'].append(velocidad)
                        tech_stats[tech]['accesos'].append(accesos)

        if len(tech_municipio_data) == 0:
            self.log("⚠️ No hay datos de tecnologías válidos")
            return None

        # Identificar las tecnologías más relevantes (por velocidad promedio)
        tech_promedios = {}
        for tech, stats in tech_stats.items():
            if stats['velocidades']:
                tech_promedios[tech] = {
                    'velocidad_promedio': np.mean(stats['velocidades']),
                    'num_municipios': len(stats['velocidades'])
                }

        # Filtrar tecnologías: las de mayor velocidad promedio Y que tengan suficientes datos
        tech_relevantes = sorted(tech_promedios.items(),
                                key=lambda x: (x[1]['num_municipios'] >= 3, x[1]['velocidad_promedio']),
                                reverse=True)[:top_tecnologias]

        tecnologias_seleccionadas = [tech[0] for tech in tech_relevantes]

        self.log(f"   Tecnologías seleccionadas: {tecnologias_seleccionadas}")

        # Filtrar datos solo para estas tecnologías
        df = pd.DataFrame(tech_municipio_data)
        df_filtrado = df[df['tecnologia'].isin(tecnologias_seleccionadas)]

        # Crear tabla pivote
        pivot_velocidad = df_filtrado.pivot_table(
            index='municipio',
            columns='tecnologia',
            values='velocidad',
            fill_value=0
        )

        # Seleccionar top municipios (por velocidad promedio across tecnologías)
        municipios_promedio = pivot_velocidad.mean(axis=1).sort_values(ascending=False)
        top_municipios_lista = municipios_promedio.head(top_municipios).index

        # Filtrar solo top municipios
        pivot_final = pivot_velocidad.loc[top_municipios_lista]

        # Crear el heatmap
        plt.figure(figsize=(max(8, len(pivot_final.columns) * 1.2),
                           max(6, len(pivot_final.index) * 0.5)))

        sns.heatmap(pivot_final,
                    annot=True,
                    cmap='viridis',
                    fmt='.0f',
                    cbar_kws={'shrink': .8, 'label': 'Velocidad (Mbps)'},
                    linewidths=0.5)

        plt.title(f'🌐 Velocidades por Tecnología\nTOP {top_municipios} Municipios × TOP {top_tecnologias} Tecnologías',
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel('Tecnología', fontsize=12)
        plt.ylabel('Municipio', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0, fontsize=10)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, f'top_tecnologias_{top_tecnologias}x{top_municipios}.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_evolucion_temporal_selectiva(self, top_municipios=12):
        """Genera evolución temporal solo para municipios más relevantes"""
        self.log(f"📊 Generando evolución temporal para TOP {top_municipios} municipios...")

        # Recopilar datos temporales
        temporal_data = []
        municipio_velocidades = {}  # Para calcular promedios

        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')

            if ('velocidades' in data and
                data['velocidades'].get('stats')):

                stats = data['velocidades']['stats']
                velocidades_municipio = []

                for stat_row in stats:
                    trimestre = stat_row.get('TRIMESTRE', 0)
                    velocidad = stat_row.get('VELOCIDAD_BAJADA_MEAN', 0)

                    if velocidad > 0:
                        temporal_data.append({
                            'municipio': municipio,
                            'trimestre': trimestre,
                            'velocidad': velocidad
                        })
                        velocidades_municipio.append(velocidad)

                if velocidades_municipio:
                    municipio_velocidades[municipio] = np.mean(velocidades_municipio)

        if len(temporal_data) == 0:
            self.log("⚠️ No hay datos temporales válidos")
            return None

        # Seleccionar top municipios por velocidad promedio
        top_municipios_lista = sorted(municipio_velocidades.items(),
                                     key=lambda x: x[1], reverse=True)[:top_municipios]
        municipios_seleccionados = [muni[0] for muni in top_municipios_lista]

        # Filtrar datos temporales
        df = pd.DataFrame(temporal_data)
        df_filtrado = df[df['municipio'].isin(municipios_seleccionados)]

        # Crear tabla pivote
        pivot_temporal = df_filtrado.pivot_table(
            index='municipio',
            columns='trimestre',
            values='velocidad',
            fill_value=0
        )

        # Ordenar municipios por velocidad promedio (descendente)
        municipios_ordenados = [muni[0] for muni in top_municipios_lista]
        pivot_temporal = pivot_temporal.reindex(municipios_ordenados)

        # Crear el heatmap
        plt.figure(figsize=(max(10, len(pivot_temporal.columns) * 0.8),
                           max(6, len(pivot_temporal.index) * 0.6)))

        sns.heatmap(pivot_temporal,
                    annot=True,
                    cmap='plasma',
                    fmt='.0f',
                    cbar_kws={'shrink': .8, 'label': 'Velocidad Bajada (Mbps)'},
                    linewidths=0.3)

        plt.title(f'📈 Evolución Temporal: TOP {top_municipios} Municipios\nVelocidad de Bajada por Trimestre',
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel('Trimestre', fontsize=12)
        plt.ylabel('Municipio', fontsize=12)
        plt.xticks(rotation=0)
        plt.yticks(rotation=0, fontsize=10)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, f'evolucion_temporal_top_{top_municipios}.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_comparativa_tecnologias_resumida(self):
        """Genera comparación de tecnologías con datos agregados"""
        self.log("📊 Generando comparativa resumida de tecnologías...")

        # Recopilar y agregar datos por tecnología
        tech_data = {}

        for data in self.datos_cargados:
            if 'tecnologias' in data and data['tecnologias'].get('trends'):
                for tech, trends in data['tecnologias']['trends'].items():
                    if tech not in tech_data:
                        tech_data[tech] = {
                            'velocidades': [],
                            'accesos': [],
                            'r2_velocidad': [],
                            'r2_accesos': []
                        }

                    # Recopilar métricas válidas
                    if trends.get('velocidad_media', 0) > 0:
                        tech_data[tech]['velocidades'].append(trends['velocidad_media'])
                    if trends.get('accesos_media', 0) > 0:
                        tech_data[tech]['accesos'].append(trends['accesos_media'])
                    if trends.get('r2_velocidad', 0) > 0:
                        tech_data[tech]['r2_velocidad'].append(trends['r2_velocidad'])
                    if trends.get('r2', 0) > 0:
                        tech_data[tech]['r2_accesos'].append(trends['r2'])

        # Calcular estadísticas por tecnología
        tech_stats = []
        for tech, data in tech_data.items():
            if data['velocidades']:  # Solo tecnologías con datos
                stats = {
                    'tecnologia': tech,
                    'velocidad_promedio': np.mean(data['velocidades']),
                    'accesos_promedio': np.mean(data['accesos']) if data['accesos'] else 0,
                    'r2_velocidad_promedio': np.mean(data['r2_velocidad']) if data['r2_velocidad'] else 0,
                    'num_municipios': len(data['velocidades'])
                }
                tech_stats.append(stats)

        if len(tech_stats) == 0:
            self.log("⚠️ No hay datos de tecnologías para comparar")
            return None

        # Crear DataFrame
        df_tech = pd.DataFrame(tech_stats)

        # Filtrar solo tecnologías con suficientes datos
        df_tech = df_tech[df_tech['num_municipios'] >= 2]

        # Preparar datos para heatmap
        metrics = ['velocidad_promedio', 'accesos_promedio', 'r2_velocidad_promedio', 'num_municipios']
        heatmap_data = df_tech[metrics].set_index(df_tech['tecnologia'])

        # Normalizar datos (0-1) para comparación visual
        heatmap_normalized = heatmap_data.div(heatmap_data.max(), axis=0).fillna(0)

        # Renombrar columnas para mejor visualización
        heatmap_data.columns = ['Velocidad Promedio\n(Mbps)', 'Accesos Promedio',
                               'Calidad Predicción\n(R²)', 'Num. Municipios']
        heatmap_normalized.columns = heatmap_data.columns

        # Crear el heatmap
        plt.figure(figsize=(10, max(6, len(heatmap_data.index) * 0.8)))

        sns.heatmap(heatmap_normalized,
                    annot=heatmap_data,  # Valores reales
                    cmap='RdYlGn',
                    fmt='.1f',
                    cbar_kws={'shrink': .8, 'label': 'Valor Normalizado (0-1)'},
                    linewidths=0.5)

        plt.title('⚖️ Comparación de Tecnologías: Métricas Promedio\n(Valores normalizados para comparación)',
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel('Métrica', fontsize=12)
        plt.ylabel('Tecnología', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, 'comparativa_tecnologias_resumida.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_mapas_inteligentes(self):
        """Función principal que genera mapas optimizados"""
        print("🧠 GENERANDO MAPAS DE CALOR INTELIGENTES")
        print("="*50)

        if not self.cargar_datos():
            print("❌ Error: No se pudieron cargar los datos")
            return []

        print(f"\n🎯 Generando mapas optimizados...")

        # Lista de mapas inteligentes a generar
        mapas_a_generar = [
            ("TOP 20 Municipios por Velocidad", lambda: self.generar_top_municipios_velocidad(20, 5)),
            ("TOP Tecnologías × Municipios", lambda: self.generar_tecnologias_principales(6, 15)),
            ("Evolución Temporal Selectiva", lambda: self.generar_evolucion_temporal_selectiva(12)),
            ("Comparativa Tecnologías", self.generar_comparativa_tecnologias_resumida)
        ]

        # Generar cada mapa
        for i, (nombre, funcion) in enumerate(mapas_a_generar, 1):
            print(f"\n📊 {i}/{len(mapas_a_generar)}. {nombre}...")
            try:
                resultado = funcion()
                if resultado:
                    print(f"   ✅ Completado")
                else:
                    print(f"   ⚠️ Sin datos suficientes")
            except Exception as e:
                print(f"   ❌ Error: {str(e)}")

        # Resumen final
        print(f"\n🎉 MAPAS INTELIGENTES COMPLETADOS!")
        print("="*50)
        print(f"📊 Total de mapas generados: {len(self.mapas_generados)}")

        if self.mapas_generados:
            print(f"\n📁 ARCHIVOS GENERADOS:")
            for i, mapa in enumerate(self.mapas_generados, 1):
                print(f"   {i:2d}. {os.path.basename(mapa)}")

            print(f"\n📂 Ubicación: {os.path.abspath(self.ruta)}")
            print(f"\n💡 Estos mapas son legibles y muestran solo los datos más relevantes!")

        return self.mapas_generados

In [3]:
# =====================================================
# FUNCIÓN PRINCIPAL PARA EJECUTAR
# =====================================================

def generar_mapas_inteligentes(carpeta='../../Limpieza/data/resultados'):
    """
    Función principal para generar mapas de calor inteligentes
    """
    generador = GeneradorMapasInteligentes(carpeta, debug=True)
    mapas = generador.generar_mapas_inteligentes()
    return mapas

# =====================================================
# PARA EJECUTAR EN JUPYTER:
# =====================================================

In [4]:
mapas_inteligentes = generar_mapas_inteligentes('../../Limpieza/data/resultados')

🧠 GENERANDO MAPAS DE CALOR INTELIGENTES
[INFO] ✅ Cargados 1121 archivos

📊 ANÁLISIS DE DATOS DISPONIBLES:
   • Total municipios: 1121
   • Con datos de velocidades: 1119
   • Con datos de tecnologías: 1118
   • Tecnologías encontradas: 16
   • Tecnologías: ['CABLE', 'FIBER TO THE ANTENNA (FTTA)', 'FIBER TO THE BUILDING O FIBER TO THE BASEMENT (FTTB)', 'FIBER TO THE CABINET (FTTC)', 'FIBER TO THE HOME (FTTH)', 'FIBER TO THE NODE (FTTN)', 'FIBER TO THE PREMISES', 'HYBRID FIBER COAXIAL (HFC)', 'NA (NO APLICA)', 'OTRAS TECNOLOGÍAS DE FIBRA (ANTES FTTX)', 'OTRAS TECNOLOGÍAS FIJAS', 'OTRAS TECNOLOGÍAS INALÁMBRICAS', 'SATELITAL', 'WIFI', 'WIMAX', 'XDSL']
   • Velocidad promedio: 169.4 Mbps
   • Rango velocidades: 4.5 - 5835.1 Mbps

💡 SUGERENCIAS DE VISUALIZACIÓN:
   • Demasiados municipios (1119) - usaremos TOP 20
   • Muchas tecnologías (16) - agruparemos similares

🎯 Generando mapas optimizados...

📊 1/4. TOP 20 Municipios por Velocidad...
[INFO] 📊 Generando TOP 20 municipios por velocidad.

/tmp/ipykernel_116971/1808217120.py:163: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_116971/1808217120.py:167: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig(plot_path, bbox_inches='tight', dpi=300)


[INFO] ✅ Guardado: top_20_municipios_velocidad.png
   ✅ Completado

📊 2/4. TOP Tecnologías × Municipios...
[INFO] 📊 Generando mapa: TOP 6 tecnologías x TOP 15 municipios...
[INFO]    Tecnologías seleccionadas: ['OTRAS TECNOLOGÍAS DE FIBRA (ANTES FTTX)', 'FIBER TO THE PREMISES', 'WIMAX', 'FIBER TO THE NODE (FTTN)', 'WIFI', 'FIBER TO THE HOME (FTTH)']


/tmp/ipykernel_116971/1808217120.py:263: UserWarning: Glyph 127760 (\N{GLOBE WITH MERIDIANS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_116971/1808217120.py:267: UserWarning: Glyph 127760 (\N{GLOBE WITH MERIDIANS}) missing from font(s) DejaVu Sans.
  plt.savefig(plot_path, bbox_inches='tight', dpi=300)


[INFO] ✅ Guardado: top_tecnologias_6x15.png
   ✅ Completado

📊 3/4. Evolución Temporal Selectiva...
[INFO] 📊 Generando evolución temporal para TOP 12 municipios...
   ❌ Error: 'str' object has no attribute 'get'

📊 4/4. Comparativa Tecnologías...
[INFO] 📊 Generando comparativa resumida de tecnologías...
   ❌ Error: `data` and `annot` must have same shape.

🎉 MAPAS INTELIGENTES COMPLETADOS!
📊 Total de mapas generados: 2

📁 ARCHIVOS GENERADOS:
    1. top_20_municipios_velocidad.png
    2. top_tecnologias_6x15.png

📂 Ubicación: /home/kingkold/data-projects/InternetAccessColombia/InternetAccessColombia-DataCleaning/Limpieza/data/resultados

💡 Estos mapas son legibles y muestran solo los datos más relevantes!


<Figure size 1000x1280 with 0 Axes>